# 01 — BCRD Exchange Rates | Bronze Ingestion (incremental, Auto Loader)

**Project:** RD Economic & Financial Intelligence Platform

## Objective
Ingest the official USD/DOP daily exchange-rate history published by the Central Bank of the Dominican Republic (BCRD) into the Bronze layer, preserving source values and adding ingestion metadata for traceability.

## Data Source (verified)
| Item | Value |
|---|---|
| Institution | Banco Central de la República Dominicana (BCRD) |
| File | `BCRD - Histórico diario de tasas.xlsx` |
| Sheet | `Histórico` (data) · `Calculo de apre-depre` (methodology notes) |
| Coverage | 20/9/2021 – 20/9/2026 (last observation 2026-09-18) |
| Layout | 8 header/notes rows; column headers on row 9; 1,253 data rows |
| Grain | One row per business day (buy and sell rate) |

## Lakehouse
- **Raw archive:** `/Volumes/rd_economic_intelligence/bronze/bcrd_raw/exchange_rates/<yyyy>/<mm>/` — one folder per arrival month, files never overwritten.
- **File register:** `rd_economic_intelligence.bronze.bcrd_exchange_rates_files` — **Auto Loader** (`binaryFile`, `trigger(availableNow=True)`, shared helper `src/shared/ingestion.py`) records each new workbook once.
- **Target table:** `rd_economic_intelligence.bronze.bcrd_exchange_rates_daily` — one snapshot per archived workbook.
- **Why keep every snapshot:** each BCRD file covers only the last five years. Keeping all files lets the history grow beyond that window; Silver merges them (newest file wins for each date).

## Bronze rules
- Source values are **not** modified, corrected or discarded.
- Column names are converted to Delta-safe snake_case (the source headers contain `|`, `%` and spaces). The mapping is documented in this notebook.
- No business date is built here; `Año/Mes/Día` stay as delivered. Date construction belongs to Silver.
- Data-quality checks are **observational** in Bronze: they report, they do not filter.

**Status:** Development — not production-ready.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
# Step 0 — Dependencies: serverless does not ship openpyxl, required by pandas to read .xlsx
dbutils.library.restartPython()

In [0]:
# ============================================================
# Step 1 — Configuration and Auto Loader file discovery
# ============================================================
from datetime import datetime, timezone
import sys, uuid, json, importlib
import pandas as pd
from pyspark.sql import functions as F

nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + nb.rsplit("/", 2)[0] + "/shared")
import ingestion as ING
importlib.reload(ING)

CATALOG = "rd_economic_intelligence"
RAW_PATH = f"/Volumes/{CATALOG}/bronze/bcrd_raw/exchange_rates/"
CHECKPOINT = f"/Volumes/{CATALOG}/bronze/bcrd_raw/_checkpoints/exchange_rates_files/"
FILES_TABLE = f"{CATALOG}.bronze.bcrd_exchange_rates_files"
TARGET_TABLE = f"{CATALOG}.bronze.bcrd_exchange_rates_daily"
SOURCE_SHEET = "Histórico"
HEADER_ROW_INDEX = 8          # 0-based: headers are on Excel row 9

# Source header -> Bronze column (Delta-safe). Order matches the source file.
COLUMN_MAPPING = {
    "Año": "anio",
    "Mes": "mes",
    "Día": "dia",
    "Compra": "compra",
    "Compra | Apre/depre acumulada (%)": "compra_apre_depre_acumulada_pct",
    "Compra | Apre/depre interanual (%)": "compra_apre_depre_interanual_pct",
    "Venta": "venta",
    "Venta | Apre/depre acumulada (%)": "venta_apre_depre_acumulada_pct",
    "Venta | Apre/depre interanual (%)": "venta_apre_depre_interanual_pct",
}
BATCH_ID, INGESTION_TS = str(uuid.uuid4()), datetime.now(timezone.utc)

registered = ING.register_new_files(spark, RAW_PATH, "*.xlsx", CHECKPOINT, FILES_TABLE,
    comment="Bronze: register of BCRD daily exchange-rate workbooks found in the raw archive by Auto Loader (one row per file).")
pending = ING.pending_files(spark, FILES_TABLE, TARGET_TABLE)
print(f"Batch ID: {BATCH_ID} · Auto Loader registered {registered} new file(s) · pending capture: {[f['_source_path'] for f in pending]}")

In [0]:
# ============================================================
# Step 2 — Per pending file: read, structural contract, observational checks, append
# ============================================================
# Bronze rules: values are not modified; checks report, they do not filter.
captured = {}
for f in pending:
    path = ING.local_path(f["_source_path"])
    title_block = pd.read_excel(path, sheet_name=SOURCE_SHEET, header=None, nrows=3)
    report_period = str(title_block.iloc[2, 0])
    pdf = pd.read_excel(path, sheet_name=SOURCE_SHEET, header=HEADER_ROW_INDEX)

    actual = [str(c).strip() for c in pdf.columns]
    missing = [c for c in COLUMN_MAPPING if c not in actual]
    unexpected = [c for c in actual if c not in COLUMN_MAPPING]
    if missing or unexpected or pdf.empty:
        raise ValueError(f"{f['_source_path']}: source layout changed. Missing: {missing} | Unexpected: {unexpected} | rows: {len(pdf)}")

    pdf.columns = actual
    pdf = pdf.rename(columns=COLUMN_MAPPING)
    pdf["_source_row_number"] = range(HEADER_ROW_INDEX + 2, HEADER_ROW_INDEX + 2 + len(pdf))
    df = ING.lineage(spark.createDataFrame(pdf), f, BATCH_ID, INGESTION_TS) \
        .withColumn("_source_sheet", F.lit(SOURCE_SHEET)).withColumn("_report_period", F.lit(report_period))

    date_col = F.make_date("anio", "mes", "dia")
    checks = {
        "row_count": df.count(),
        "invalid_calendar_date": df.filter(date_col.isNull()).count(),
        "duplicate_dates": df.groupBy("anio", "mes", "dia").count().filter("count > 1").count(),
        "non_positive_rates": df.filter((F.col("compra") <= 0) | (F.col("venta") <= 0)).count(),
        "buy_greater_than_sell": df.filter(F.col("compra") > F.col("venta")).count(),
    }
    for name, value in checks.items():
        print(f"  {'INFO' if name == 'row_count' else ('PASS' if value == 0 else 'WARN'):<4} {name}: {value:,}")

    df.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)  # one atomic append per file
    loaded = spark.table(TARGET_TABLE).filter((F.col("_batch_id") == BATCH_ID) & (F.col("_source_path") == f["_source_path"])).count()
    assert loaded == checks["row_count"], f"Row count mismatch: read {checks['row_count']}, loaded {loaded}"
    captured[f["_source_path"].split("exchange_rates/")[-1]] = {"rows": loaded, "report_period": report_period, "dq_checks": checks}

spark.sql(f"COMMENT ON TABLE {TARGET_TABLE} IS 'Bronze: BCRD official USD/DOP daily buy/sell rates, one snapshot per archived workbook, source-preserving, with lineage.'") if spark.catalog.tableExists(TARGET_TABLE) else None

In [0]:
summary = spark.sql(f"""SELECT COUNT(DISTINCT _source_path) AS snapshots, COUNT(*) AS rows,
                        MIN(make_date(anio, mes, dia)) AS first_date, MAX(make_date(anio, mes, dia)) AS last_date FROM {TARGET_TABLE}""").first()
dbutils.notebook.exit(json.dumps({"status": "SUCCESS" if captured else "NO_NEW_FILES", "registered_by_auto_loader": registered,
    "captured": captured, "snapshots_in_bronze": summary["snapshots"], "first_date": str(summary["first_date"]), "last_date": str(summary["last_date"])}, ensure_ascii=False))